In [1]:
import chromadb
import pandas as pd

from embeders import OllamaEmbeddingFunction

from tqdm import tqdm

In [2]:
df_checks = pd.read_excel("./Analise manual Aos fatos.xlsx")
client = chromadb.PersistentClient(path="./.chroma_db")
ef=OllamaEmbeddingFunction()
collection = client.get_or_create_collection(name="checks_aos_fatos", embedding_function=ef)


# Embed and save checks

In [ ]:
# for idx, link, resumo in df_checks[df_checks['Resumo'].notna()][['Link', 'Resumo']].to_records():
#     collection.upsert(
#         ids=[str(idx)],
#         documents=[resumo],
#         metadatas=[{"link": link}]
#     )


# query

In [3]:
df_pubs = pd.read_excel("./2023_Completo_redem_0304.xlsx")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 64  # You can adjust this based on your resources

def process_batch(batch_df):
    messages = batch_df["Message"].tolist()
    res = collection.query(query_texts=messages, n_results=3)
    results = []
    for idx, message in enumerate(messages):
        distances = res["distances"][idx]
        min_distance = min(distances)
        results.append({
            "message": message,
            "min_distance": min_distance,
            "min_dist_document": res["documents"][idx][0],
            "doc_id": res["ids"][idx][0],
            "pub_id": batch_df.iloc[idx]["Message-ID"],
            "all_distances": res["distances"][idx],
            "all_documents": res["documents"][idx],
            "all_ids": res["ids"][idx],
        })
    return results

print("Iniciando ...")
df_res_list = []
batches = [df_pubs.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_pubs), BATCH_SIZE)]

print(f"Total de mensagens: {len(df_pubs)}")
print(f"Total de batches: {len(batches)}")
print("Iniciando processamento...")
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]
    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing batches"):
        df_res_list.extend(f.result())
        df_res = pd.DataFrame(df_res_list)
        df_res.to_csv("df_res.csv", index=False)


Iniciando ...
Total de mensagens: 370395
Total de batches: 5788
Iniciando processamento...


Processing batches:   1%|          | 32/5788 [20:57<72:34:11, 45.39s/it]